# MPRA evaluation: a tolerant metric, and the CAV's own confidence

Two follow-ups from the 2026-10-27 meeting. **CPU only**: everything here reads results already on Drive (the hits,
the SNVs, and every model's function-CAV fits). No model is loaded.

**A. Do not demand base-for-base alignment (Rong's suggestion).** Every score so far correlates the prediction and the
measured log2FC SNV by SNV. If the measured dip sits at bases 280–285 and the prediction at 282–287, that 2 bp offset
is punished as hard as predicting the wrong site. Two ways to stop doing that:

* **Smoothed correlation.** Average both tracks over a sliding window of *w* bases first (w = 1, 3, 5, 11, 21),
  then correlate. w = 1 is the per-base correlation.
* **Hits and misses with a tolerance of ±k bp.** Call the strongest 15% of bases by |measured log2FC| the *measured
  effects*. A predicted base is *right* if a measured effect of the same sign lies within k bases of it; a measured
  effect is *found* if a prediction of the same sign lies within k bases. Precision = share of predictions that are
  right; recall = share of measured effects found. motifbreakR should have high recall and low precision (it calls
  many motifs); the CAV should be the other way round. The precision-recall curve shows both at every threshold.

**B. Use the CAV's own confidence (Jingyuan's suggestion).** The CAV says how sure it is, without looking at the
MPRA: P(bound) of the motif it picks, and the held-out AUC of that motif's direction (how well the direction
separates ChIP-bound from unbound sites). Three questions:

1. Over (region, model) pairs: where the CAV is more confident on average, is its gain over motifbreakR larger?
2. Calibration: are SNVs where the CAV is confident really the ones with bigger measured effects?
3. On the confident SNVs only, does the CAV's choice of motif beat motifbreakR's?

**Same predictions as before.** The motifbreakR and motifbreakR + CAV code is copied unchanged from
`Function CAV - Cross-Model, Region 0.ipynb` (section 3.3), which reproduced `Hit Selection` exactly. Section 1.3
checks that again on all three regions before anything new is computed.

In [ ]:
!pip install -q scipy pandas matplotlib
from google.colab import drive
drive.mount("/content/drive")

## 1.1 Settings

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REGIONS = [0, 8, 20]
SRC_DIR = {0: "fcav_pgc_hk_r0", 8: "fcav_pgc_hk", 20: "fcav_pgc_hk_r20"}          # hits, SNVs, MPRA windows
LEGACY = {0: "fcav_pgc_hk_lr_r0", 8: "fcav_pgc_hk_lr", 20: "fcav_pgc_hk_lr_r20"}   # NT-v2 500M directions
XM = f"{WORK_DIR}/fcav_xm"                  # other models: fcav_xm/<model>_r<region>/fits
OUT = f"{WORK_DIR}/mpra_eval"
os.makedirs(OUT, exist_ok=True)

LP_FIX = {"nt500m": 24}     # layer per model; NT-v2 500M at L24 as in Hit Selection, every other model its best-AUC layer
MID, K = 0.75, 20           # w(P) = sigmoid(K (P - MID)), as in Hit Selection
MBR_THRESH, MBR_NEUT = 0.85, 0.4

WINDOWS = [1, 3, 5, 11, 21] # smoothing widths in bp
W_MAIN = 5                  # the width reported in the summary table
TOL = [0, 2, 5]             # tolerance in bp for hits and misses
K_MAIN = 2
EFFECT_TOP = 0.15           # "measured effect" = the strongest 15% of bases by |mean log2FC|
PRED_MIN = 0.1              # a base counts as "predicted" when |prediction| >= this (motifbreakR reports |alleleDiff| >= 0.4)
P_CONF = 0.75               # "confident" = P(bound) of the CAV-chosen motif >= this (the centre of the gate)
N_PERM = 1000

C = json.load(open(f"{WORK_DIR}/config.json"))
INK = "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": INK,
                     "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK})
pd.set_option("display.width", 250)

## 1.2 Load every region and every model's P(bound)

The motifbreakR scoring and the two rules are copied from the cross-model notebook (3.3):

| rule | motif used at each SNV | prediction |
|---|---|---|
| motifbreakR | largest \|alleleDiff\| | alleleDiff |
| CAV-chosen motifbreakR | highest P(bound) | alleleDiff |
| motifbreakR + CAV (the one shown at the meetings) | highest P(bound) | alleleDiff × w(P) |

In [ ]:
wfun = lambda x: 1 / (1 + np.exp(-K * (x - MID)))


def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            m = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(m.group(1)) if m else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENC = {"A": 0, "C": 1, "G": 2, "T": 3}


def load_fits(path):
    # P(bound) per hit x layer, held-out AUC per motif x layer, total depth
    dec, auc, nl = [], {}, None
    for f in sorted(os.listdir(path)):
        if not f.endswith(".npz"):
            continue
        z = np.load(f"{path}/{f}", allow_pickle=False)
        if "dec_hit" not in z or not len(z["hit_id"]):
            continue
        lids = [int(l) for l in (z["layers"] if "layers" in z else np.arange(z["dec_hit"].shape[1]))]
        dec.append(pd.DataFrame(z["dec_hit"], index=z["hit_id"], columns=lids))
        auc[f[:-4]] = pd.Series(np.asarray(z["auc"], float), index=lids)
        nl = int(z["n_layers"]) if "n_layers" in z else z["dec_hit"].shape[1] - 1
    return 1 / (1 + np.exp(-pd.concat(dec))), pd.DataFrame(auc).T, nl


def region_runs(r):
    runs = {"nt500m": f"{WORK_DIR}/{LEGACY[r]}/fits"}
    if os.path.isdir(XM):
        for d in sorted(os.listdir(XM)):
            if d.endswith(f"_r{r}") and os.path.isdir(f"{XM}/{d}/fits"):
                tag = d[: -len(f"_r{r}")]
                runs["nt500m (new readout)" if tag == "nt500m" else tag] = f"{XM}/{d}/fits"
    return runs


def build(r, HS):
    # SNV x hit rows and motifbreakR pairs on the hits HS (verbatim logic of the cross-model notebook, 3.3)
    S_ = f"{WORK_DIR}/{SRC_DIR[r]}"
    MWJ = json.load(open(f"{S_}/mpra_windows.json"))
    hits = pd.read_csv(f"{S_}/hits.csv").set_index("hit_id")
    snv = pd.read_csv(f"{S_}/snvs.csv")
    q = pd.read_csv(f"{S_}/snv_hit_dpwm.csv")
    q = q[q["hit_id"].isin(HS)].sort_values(["snv", "hit_id"]).reset_index(drop=True)
    sn = q["snv"].values
    st = np.r_[0, np.flatnonzero(sn[1:] != sn[:-1]) + 1]
    hi = np.searchsorted(HS, q["hit_id"].values)
    n_snv = len(st)
    MOT = hits.loc[HS, "motif"].values
    base = {}
    for h in MWJ["hits"]:
        for i, b in enumerate(h["window"]):
            base.setdefault(h["start"] - C["motif_offset"] + i, b)
    lo = min(base)
    seq = np.array([ENC.get(base.get(x, "N"), 4) for x in range(lo, max(base) + 1)])
    s_ = snv.loc[sn[st]]
    pos, ref, alt = s_["local"].values - lo, s_["ref"].map(ENC).values, s_["alt"].map(ENC).values
    assert (seq[pos] == ref).all(), "rebuilt sequence disagrees with the SNV reference bases"
    out = []
    for m in sorted(set(MOT)):
        if m not in MBR_MATS:
            continue
        S, Sr, smin, smax = MBR_MATS[m]
        w = len(S)
        n = len(seq) - w + 1
        idx = np.arange(n)[:, None] + np.arange(w)[None, :]
        f_fwd, f_rev = S[np.arange(w), seq[idx]].sum(1), Sr[np.arange(w), seq[idx]].sum(1)
        stw = pos[:, None] - np.arange(w)[None, :]
        k = np.broadcast_to(np.arange(w), stw.shape)
        ok = (stw >= 0) & (stw < n)
        stc = np.clip(stw, 0, n - 1)
        ref_f, ref_r = np.where(ok, f_fwd[stc], -np.inf), np.where(ok, f_rev[stc], -np.inf)
        alt_f = ref_f + np.where(ok, S[k, alt[:, None]] - S[k, ref[:, None]], 0)
        alt_r = ref_r + np.where(ok, Sr[k, alt[:, None]] - Sr[k, ref[:, None]], 0)
        sr, sa = np.maximum(ref_f, ref_r).max(1), np.maximum(alt_f, alt_r).max(1)
        pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
        diff = sa - sr
        keep = np.isfinite(sr) & ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
        out += [(i, m, diff[i]) for i in np.flatnonzero(keep)]
    t = pd.DataFrame(out, columns=["i", "motif", "alleleDiff"])
    t["j"] = np.arange(len(t))
    rows_ = pd.DataFrame({"i": np.repeat(np.arange(n_snv), np.diff(np.r_[st, len(sn)])), "motif": MOT[hi], "h": hi})
    link = t[["j", "i", "motif"]].merge(rows_, on=["i", "motif"])
    keep = np.sort(link["j"].unique())
    t = t.loc[keep].reset_index(drop=True)
    # every base of the region with its mean measured log2FC; SNVs outside the hits get prediction 0
    pos_all = np.arange(snv["local"].min(), snv["local"].max() + 1)
    meas = snv.groupby("local")["log2FC"].mean().reindex(pos_all).values
    return dict(r=r, HS=HS, MOT=MOT, hits=hits, snv=snv, n_snv=n_snv, snv_id=sn[st],
                y=snv.loc[sn[st], "log2FC"].values, t=t, lj=np.searchsorted(keep, link["j"].values),
                lh=link["h"].values, DIFF=t["alleleDiff"].values, TI=t["i"].values, MOTP=t["motif"].values,
                GROUPS=[np.flatnonzero(MOT == m) for m in np.unique(MOT)], pos_all=pos_all, meas=meas)


def predict(R, p_hit):
    # per SNV: the three rules, plus the P(bound) and motif of the CAV's choice and motifbreakR's |alleleDiff|
    P = np.full(len(R["t"]), -np.inf)
    np.maximum.at(P, R["lj"], p_hit[R["lh"]])
    TI, DIFF, n = R["TI"], R["DIFF"], R["n_snv"]

    def first(key):
        o = np.lexsort((-key, TI))
        return o[np.r_[True, TI[o][1:] != TI[o][:-1]]]

    fb, fc = first(np.abs(DIFF)), first(P + 1e-9 * np.abs(DIFF))
    out = {k: np.zeros(n) for k in ("motifbreakR", "CAV-chosen motifbreakR", "motifbreakR + CAV")}
    out["motifbreakR"][TI[fb]] = DIFF[fb]
    out["CAV-chosen motifbreakR"][TI[fc]] = DIFF[fc]
    out["motifbreakR + CAV"][TI[fc]] = DIFF[fc] * wfun(P[fc])
    out["P chosen"] = np.full(n, np.nan); out["P chosen"][TI[fc]] = P[fc]
    out["motif chosen"] = np.full(n, None, dtype=object); out["motif chosen"][TI[fc]] = R["MOTP"][fc]
    out["|alleleDiff| mbR"] = np.full(n, np.nan); out["|alleleDiff| mbR"][TI[fb]] = np.abs(DIFF[fb])
    return out


def perm_motif(R, p, rng):
    p = p.copy()
    for g_ in R["GROUPS"]:
        p[g_] = p[g_][rng.permutation(len(g_))]
    return p


# ---------- load ----------
PB, AUCM, NL, CTX, LAYER = {}, {}, {}, {}, {}
for r in REGIONS:
    for tag, p in region_runs(r).items():
        if os.path.isdir(p) and any(f.endswith(".npz") for f in os.listdir(p)):
            PB[r, tag], AUCM[r, tag], NL[r, tag] = load_fits(p)
    tags = [tg for (rr, tg) in PB if rr == r]
    sets = [set(PB[r, tg].index) for tg in tags]
    HS = np.array(sorted(set.intersection(*sets)))     # only the hits every model has a direction for, as at the meeting
    CTX[r] = build(r, HS)
    for tg in tags:
        LAYER[r, tg] = LP_FIX.get(tg, int(AUCM[r, tg].mean().idxmax()))
    R = CTX[r]
    print(f"region {r:2d}: {len(HS)} hits shared by {', '.join(f'{tg} L{LAYER[r, tg]}/{NL[r, tg]}' for tg in tags)}; "
          f"{R['n_snv']} SNVs inside them, {R['t']['i'].nunique()} with a motifbreakR call; "
          f"{len(R['pos_all'])} bases in the region")
MODELS = sorted({tg for (_, tg) in PB}, key=lambda x: (x != "nt500m", x))
COL = {tg: f"C{i}" for i, tg in enumerate(MODELS)}

## 1.3 Check: the old numbers come back

NT-v2 500M alone, on **all** its hits (no sharing with other models), L24, SNV by SNV. These must equal the 2026-10-20
write-up (section 5.3): Spearman |·| 0.120 / 0.333 / 0.298 → 0.174 / 0.407 / 0.323 and Pearson 0.177 / 0.279 / 0.464 →
0.404 / 0.625 / 0.579 for regions 0 / 8 / 20. If they do not, stop: the inputs on Drive have changed.

In [ ]:
def per_snv(pred, y):
    if np.std(pred) == 0:
        return np.array([np.nan] * 3)
    return np.array([stats.spearmanr(pred, y)[0], stats.spearmanr(np.abs(pred), np.abs(y))[0], stats.pearsonr(pred, y)[0]])


WANT = {"motifbreakR": {0: (0.120, 0.177), 8: (0.333, 0.279), 20: (0.298, 0.464)},
        "motifbreakR + CAV": {0: (0.174, 0.404), 8: (0.407, 0.625), 20: (0.323, 0.579)}}
rows, worst = [], 0
for r in REGIONS:
    Pn = PB[r, "nt500m"]
    R0 = build(r, np.array(sorted(Pn.index)))
    pr = predict(R0, Pn.loc[R0["HS"], 24].values)
    for rule, want in WANT.items():
        s = per_snv(pr[rule], R0["y"])
        rows.append({"region": r, "rule": rule, "Spearman |.|": s[1], "want": want[r][0], "Pearson": s[2], "want ": want[r][1]})
        worst = max(worst, abs(s[1] - want[r][0]), abs(s[2] - want[r][1]))
print(pd.DataFrame(rows).set_index(["region", "rule"]).to_string(float_format=lambda x: f"{x:.3f}"))
print(f"\nlargest difference: {worst:.4f}" + ("   OK" if worst < 0.002 else "   <-- DOES NOT REPRODUCE, check the inputs"))

---
# Part A — a metric that tolerates a few bases of offset

## A.1 The metric functions

In [ ]:
def track(R, pred):
    # per-SNV prediction (SNVs inside the hits) -> mean per base over the whole region; 0 where nothing is predicted
    full = pd.Series(0.0, index=R["snv"].index)
    full.loc[R["snv_id"]] = pred
    return full.groupby(R["snv"]["local"]).mean().reindex(R["pos_all"], fill_value=0.0).values


def smooth(a, w):
    return a if w == 1 else pd.Series(a).rolling(w, center=True, min_periods=1).mean().values


def window_scores(meas, pred, w):
    ok = np.isfinite(meas)
    m, p = smooth(meas, w)[ok], smooth(pred, w)[ok]
    if np.std(p) == 0:
        return np.array([np.nan] * 3)
    return np.array([stats.pearsonr(p, m)[0], stats.spearmanr(p, m)[0], stats.spearmanr(np.abs(p), np.abs(m))[0]])


def dilate(b, k):
    return b if k == 0 else np.convolve(b.astype(float), np.ones(2 * k + 1), "same") > 0


def effects(meas):
    ok = np.isfinite(meas)
    m = np.where(ok, meas, 0.0)
    thr = np.quantile(np.abs(m[ok]), 1 - EFFECT_TOP)
    return ok & (m <= -thr), ok & (m >= thr), ok


def agreement(meas, pred, k, pred_min=PRED_MIN, E=None):
    En, Ep, ok = E if E is not None else effects(meas)
    Pn, Pp = ok & (pred <= -pred_min), ok & (pred >= pred_min)
    n_pred, n_eff = (Pn | Pp).sum(), (En | Ep).sum()
    right = ((Pn & dilate(En, k)) | (Pp & dilate(Ep, k))).sum()
    found = ((En & dilate(Pn, k)) | (Ep & dilate(Pp, k))).sum()
    prec = right / n_pred if n_pred else np.nan
    rec = found / n_eff
    f1 = 2 * prec * rec / (prec + rec) if n_pred and prec + rec > 0 else 0.0
    return prec, rec, f1, int(n_pred)


# a 10-base toy: the measured dip is at bases 3-5, the prediction at 5-7 (2 bases off)
_m = np.array([0, 0, 0, -1, -1, -1, 0, 0, 0, 0.])
_p = np.array([0, 0, 0, 0, 0, -1, -1, -1, 0, 0.])
_E = (_m <= -0.5, _m >= 0.5, np.isfinite(_m))
print("toy: per-base Pearson %.2f;" % stats.pearsonr(_p, _m)[0],
      "  precision / recall with k = 0: %.2f / %.2f;" % agreement(_m, _p, 0, E=_E)[:2],
      "  with k = 2: %.2f / %.2f" % agreement(_m, _p, 2, E=_E)[:2])

## A.2 Every region, every model

For each region and model: motifbreakR and motifbreakR + CAV, scored SNV by SNV (the old way), smoothed at every
width, and by hits and misses at every tolerance. The CAV rows get the same two nulls as before (P(bound) shuffled
across hits, N1, and within motif, N2) for the three headline numbers.

In [ ]:
RULES = ["motifbreakR", "CAV-chosen motifbreakR", "motifbreakR + CAV"]
rng = np.random.default_rng(5)
rows, PRED = [], {}


def headline(R, pr):
    tr = track(R, pr)
    return np.array([per_snv(pr, R["y"])[2], window_scores(R["meas"], tr, W_MAIN)[0], agreement(R["meas"], tr, K_MAIN, E=R["E"])[2]])


for r in REGIONS:
    R = CTX[r]
    R["E"] = effects(R["meas"])
    for tag in MODELS:
        if (r, tag) not in PB:
            continue
        lp = LAYER[r, tag]
        p = PB[r, tag].loc[R["HS"], lp].values
        pr = predict(R, p)
        PRED[r, tag] = pr
        n1 = np.array([headline(R, predict(R, rng.permutation(p))["motifbreakR + CAV"]) for _ in range(N_PERM)])
        n2 = np.array([headline(R, predict(R, perm_motif(R, p, rng))["motifbreakR + CAV"]) for _ in range(N_PERM)])
        for rule in RULES:
            if rule == "motifbreakR" and tag != MODELS[0]:
                continue                                   # the same for every model (shared hits): keep one copy
            tr = track(R, pr[rule])
            row = {"region": r, "model": "-" if rule == "motifbreakR" else f"{tag} L{lp}", "tag": tag, "rule": rule}
            row.update(dict(zip(["SNV: Spearman signed", "SNV: Spearman |.|", "SNV: Pearson"], per_snv(pr[rule], R["y"]))))
            for w in WINDOWS:
                row.update(dict(zip([f"w{w}: Pearson", f"w{w}: Spearman", f"w{w}: Spearman |.|"], window_scores(R["meas"], tr, w))))
            for k in TOL:
                row.update(dict(zip([f"±{k}: precision", f"±{k}: recall", f"±{k}: F1", "bases predicted"],
                                    agreement(R["meas"], tr, k, E=R["E"]))))
            if rule == "motifbreakR + CAV":
                obs = headline(R, pr[rule])
                for j, nm in enumerate(["SNV: Pearson", f"w{W_MAIN}: Pearson", f"±{K_MAIN}: F1"]):
                    row[f"{nm} p N1"] = np.mean(n1[:, j] >= obs[j])
                    row[f"{nm} p N2"] = np.mean(n2[:, j] >= obs[j])
            rows.append(row)
    print(f"region {r} done")
EV = pd.DataFrame(rows)
EV.to_csv(f"{OUT}/scores.csv", index=False)

show = ["SNV: Pearson", f"w{W_MAIN}: Pearson", f"w{W_MAIN}: Spearman |.|",
        f"±{K_MAIN}: precision", f"±{K_MAIN}: recall", f"±{K_MAIN}: F1", "bases predicted",
        "SNV: Pearson p N1", f"w{W_MAIN}: Pearson p N1", f"±{K_MAIN}: F1 p N1"]
print(f"\nsummary: smoothing width {W_MAIN} bp, tolerance ±{K_MAIN} bp, measured effects = top {EFFECT_TOP:.0%} of bases, "
      f"predicted = |prediction| >= {PRED_MIN}; p = share of {N_PERM} shuffles of P(bound) at least as good\n")
print(EV.set_index(["region", "model", "rule"])[show].to_string(float_format=lambda x: f"{x:.3f}", na_rep=""))

## A.3 Does smoothing change the picture?

Correlation against the smoothing width. Dashed: motifbreakR. One line per model for motifbreakR + CAV. The left end
(w = 1) is the per-base score.

In [ ]:
OURS = "motifbreakR + CAV"
fig, ax = plt.subplots(2, len(REGIONS), figsize=(5 * len(REGIONS), 7), sharex=True, sharey="row")
for j, r in enumerate(REGIONS):
    e = EV[EV.region == r]
    for i, mt in enumerate(["Pearson", "Spearman |.|"]):
        a = ax[i, j]
        b = e[e.rule == "motifbreakR"].iloc[0]
        a.plot(WINDOWS, [b[f"w{w}: {mt}"] for w in WINDOWS], c=INK, ls="--", lw=1.6, marker="o", ms=3, label="motifbreakR")
        for _, rw in e[e.rule == OURS].iterrows():
            a.plot(WINDOWS, [rw[f"w{w}: {mt}"] for w in WINDOWS], lw=2, marker="o", ms=3, label=rw["model"], c=COL[rw["tag"]])
        a.set_xscale("log"); a.set_xticks(WINDOWS, [str(w) for w in WINDOWS])
        a.set_title(f"region {r}: {mt}", loc="left", fontsize=10)
        if i == 1:
            a.set_xlabel("smoothing width (bp)")
        if j == 0:
            a.set_ylabel(f"{mt}, prediction vs measured")
    ax[0, j].legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/smoothing.png", dpi=150, bbox_inches="tight"); plt.show()

## A.4 Precision and recall with ±2 bp tolerance

Each curve sweeps the threshold on |prediction| from high (few bases predicted, left) to low (many, right). motifbreakR
should reach further right (finds more) and the CAV should sit higher (is right more often). The dot is the operating
point at |prediction| >= 0.1. Grey line: precision of predicting every base (= share of bases that are measured effects
within the tolerance).

In [ ]:
fig, ax = plt.subplots(1, len(REGIONS), figsize=(5 * len(REGIONS), 4.2), sharey=True)
for j, r in enumerate(REGIONS):
    R, a = CTX[r], ax[j]
    curves = [("motifbreakR", PRED[r, MODELS[0]]["motifbreakR"], dict(c=INK, ls="--"))]
    curves += [(f"{tg} L{LAYER[r, tg]}", PRED[r, tg][OURS], dict(c=COL[tg])) for tg in MODELS if (r, tg) in PRED]
    for name, pr, kw in curves:
        tr = track(R, pr)
        v = np.abs(tr[np.abs(tr) > 1e-6])
        ths = np.unique(np.quantile(v, np.linspace(0, 1, 40))) if len(v) else []
        pts = np.array([agreement(R["meas"], tr, K_MAIN, pred_min=th, E=R["E"])[:2] for th in ths[::-1]])
        if len(pts):
            l, = a.plot(pts[:, 1], pts[:, 0], lw=2, label=name, **kw)
            op = agreement(R["meas"], tr, K_MAIN, E=R["E"])
            a.scatter([op[1]], [op[0]], s=40, color=l.get_color(), zorder=3)
    En, Ep, ok = R["E"]
    base = (ok & dilate(En | Ep, K_MAIN)).sum() / ok.sum()
    a.axhline(base, c="#bdbdb8", lw=1)
    a.set_xlim(0, 1); a.set_ylim(0, 1)
    a.set_xlabel(f"recall: share of measured effects found (±{K_MAIN} bp)")
    a.set_title(f"region {r}", loc="left", fontsize=10)
    a.legend(frameon=False, fontsize=7)
ax[0].set_ylabel(f"precision: share of predicted bases that are right (±{K_MAIN} bp)")
plt.tight_layout(); plt.savefig(f"{OUT}/precision_recall.png", dpi=150, bbox_inches="tight"); plt.show()

---
# Part B — the CAV's own confidence

Two confidence scores, both available without the MPRA:

* **P(bound)** of the motif the CAV picks at that SNV;
* **AUC of that motif's direction**: how well it separated ChIP-bound from unbound sites on held-out sites, at the
  layer in use.

## B.1 Is the gain larger where the CAV is more confident on average?

One point per (region, model). x: the region's mean confidence over its hits; y: motifbreakR + CAV minus motifbreakR.
Points in the same region share the same MPRA data, so they are not independent; with 3 regions this is a look, not a
test.

In [ ]:
rows = []
for (r, tag), pr in PRED.items():
    R, lp = CTX[r], LAYER[r, tag]
    p = PB[r, tag].loc[R["HS"], lp].values
    auc_hit = AUCM[r, tag][lp].reindex(R["MOT"]).values
    e = EV[EV.region == r]
    b, o = e[e.rule == "motifbreakR"].iloc[0], e[(e.rule == OURS) & (e.model == f"{tag} L{lp}")].iloc[0]
    rows.append({"region": r, "model": tag, "layer": lp, "mean AUC of the hits' motifs": np.nanmean(auc_hit),
                 "mean P(bound) of hits": p.mean(), "share of hits with P >= 0.75": np.mean(p >= P_CONF),
                 "gain SNV Pearson": o["SNV: Pearson"] - b["SNV: Pearson"],
                 f"gain w{W_MAIN} Pearson": o[f"w{W_MAIN}: Pearson"] - b[f"w{W_MAIN}: Pearson"],
                 f"gain ±{K_MAIN} F1": o[f"±{K_MAIN}: F1"] - b[f"±{K_MAIN}: F1"]})
CF = pd.DataFrame(rows).sort_values(["region", "model"])
CF.to_csv(f"{OUT}/confidence_by_region_model.csv", index=False)
print(CF.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))

XS = ["mean AUC of the hits' motifs", "share of hits with P >= 0.75"]
YS = ["gain SNV Pearson", f"gain w{W_MAIN} Pearson", f"gain ±{K_MAIN} F1"]
print("\nSpearman over the points:")
for x in XS:
    print("  " + x + ":  " + ",  ".join(f"{y_} {stats.spearmanr(CF[x], CF[y_])[0]:+.2f}" for y_ in YS))

fig, ax = plt.subplots(len(XS), len(YS), figsize=(4.6 * len(YS), 3.8 * len(XS)), sharey="col")
mk = {r: m for r, m in zip(REGIONS, "osD^v")}
for i, x in enumerate(XS):
    for j, y_ in enumerate(YS):
        a = ax[i, j]
        for _, rw in CF.iterrows():
            a.scatter(rw[x], rw[y_], marker=mk[rw["region"]], s=50, c=COL[rw["model"]])
            a.annotate(f"{rw['model']} r{rw['region']}", (rw[x], rw[y_]), fontsize=6, xytext=(3, 3), textcoords="offset points")
        a.axhline(0, c=INK, lw=.8, ls="--")
        a.set_xlabel(x, fontsize=8)
        if j == 0 or i == 0:
            a.set_ylabel(y_, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/confidence_vs_gain.png", dpi=150, bbox_inches="tight"); plt.show()

## B.2 Calibration: are confident calls the big effects?

Every SNV where the CAV makes a call, pooled over regions. The confidence score is cut into fifths (1 = least
confident). y: the measured |log2FC| as a percentile within its region (50 = typical). A useful confidence score rises
from left to right. Dashed: motifbreakR's own confidence, the |alleleDiff| of the motif it picks, on the same SNVs.

In [ ]:
def pct_within(v):
    return 100 * stats.rankdata(v) / len(v)


rows = []
for (r, tag), pr in PRED.items():
    R, lp = CTX[r], LAYER[r, tag]
    has = np.isfinite(pr["P chosen"])
    aucs = AUCM[r, tag][lp].reindex(pr["motif chosen"][has]).values
    rows.append(pd.DataFrame({"region": r, "model": tag, "|log2FC| pct": pct_within(np.abs(R["y"][has])),
                              "P(bound)": pr["P chosen"][has], "AUC": aucs, "|alleleDiff|": pr["|alleleDiff| mbR"][has]}))
CB = pd.concat(rows, ignore_index=True)


def by_fifth(d, col):
    d = d[np.isfinite(d[col])]
    f = pd.qcut(d[col].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
    return d.groupby(f, observed=True)["|log2FC| pct"].agg(["mean", "sem", "size"])


fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
summ = []
for i, conf in enumerate(["P(bound)", "AUC"]):
    a = ax[i]
    for tag in MODELS:
        d = CB[CB.model == tag]
        if len(d) < 25:
            continue
        g = by_fifth(d, conf)
        a.errorbar(g.index.astype(int), g["mean"], yerr=g["sem"], lw=2, marker="o", ms=4, capsize=2, label=tag,
                   c=COL[tag])
        summ.append({"confidence": conf, "model": tag, "SNVs": len(d),
                     "Spearman(conf, |log2FC| pct)": stats.spearmanr(d[conf], d["|log2FC| pct"], nan_policy="omit")[0],
                     "top fifth - bottom fifth": g["mean"].iloc[-1] - g["mean"].iloc[0]})
    d = CB[CB.model == MODELS[0]]
    g = by_fifth(d, "|alleleDiff|")
    a.plot(g.index.astype(int), g["mean"], c=INK, ls="--", lw=1.6, marker="o", ms=3, label="motifbreakR |alleleDiff|")
    a.axhline(50, c="#bdbdb8", lw=1)
    a.set_xticks([1, 2, 3, 4, 5]); a.set_xlabel(f"fifth of the CAV's {conf} (1 = least confident)")
    a.set_title(f"confidence = {conf} of the CAV-chosen motif", loc="left", fontsize=10)
    a.legend(frameon=False, fontsize=7)
ax[0].set_ylabel("measured |log2FC|, percentile within region")
plt.tight_layout(); plt.savefig(f"{OUT}/calibration.png", dpi=150, bbox_inches="tight"); plt.show()
print(pd.DataFrame(summ).to_string(index=False, float_format=lambda x: f"{x:+.3f}"))

## B.3 On the SNVs where the CAV is confident, does its choice of motif beat motifbreakR's?

SNVs split by the P(bound) of the CAV-chosen motif (>= 0.75 = confident). Both rules use the same alleleDiff scale;
they differ only in which motif they read it from, so this compares the two choices. (The w(P) gate is left out here:
on the low-confidence SNVs it would shrink the CAV towards 0 by construction.)

In [ ]:
rows = []
for (r, tag), pr in PRED.items():
    R = CTX[r]
    has = np.isfinite(pr["P chosen"])
    for name, sel in [(f"confident (P >= {P_CONF})", has & (pr["P chosen"] >= P_CONF)),
                      (f"not confident (P < {P_CONF})", has & (pr["P chosen"] < P_CONF))]:
        if sel.sum() < 20:
            rows.append({"region": r, "model": tag, "SNVs": name, "n": int(sel.sum())}); continue
        y_ = R["y"][sel]
        sb, sc = per_snv(pr["motifbreakR"][sel], y_), per_snv(pr["CAV-chosen motifbreakR"][sel], y_)
        same = np.mean(pr["motifbreakR"][sel] == pr["CAV-chosen motifbreakR"][sel])
        rows.append({"region": r, "model": tag, "SNVs": name, "n": int(sel.sum()), "same motif": same,
                     "mbR Spearman |.|": sb[1], "CAV Spearman |.|": sc[1], "gain |.|": sc[1] - sb[1],
                     "mbR Pearson": sb[2], "CAV Pearson": sc[2], "gain Pearson": sc[2] - sb[2]})
CS = pd.DataFrame(rows)
CS.to_csv(f"{OUT}/confident_vs_not.csv", index=False)
print(CS.set_index(["region", "model", "SNVs"]).to_string(float_format=lambda x: f"{x:+.3f}", na_rep=""))
print("\n'same motif' = share of these SNVs where the two rules read the same alleleDiff (then they cannot differ)")

## End

Outputs in `mpra_eval/`: `scores.csv` (every metric, region, model and rule), `confidence_by_region_model.csv`,
`confident_vs_not.csv`, and the four figures.